<a href="https://colab.research.google.com/github/kamil0sek1/kursAI/blob/main/Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import pandas as pd # tabele danych
import seaborn as sns # źródło danych
import matplotlib.pyplot as plt # wykresy
from scipy import stats


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


import numpy as np

In [31]:
titanic = sns.load_dataset("titanic")
print(titanic.head(10))

   survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
0         0       3    male  22.0      1      0   7.2500        S   Third   
1         1       1  female  38.0      1      0  71.2833        C   First   
2         1       3  female  26.0      0      0   7.9250        S   Third   
3         1       1  female  35.0      1      0  53.1000        S   First   
4         0       3    male  35.0      0      0   8.0500        S   Third   
5         0       3    male   NaN      0      0   8.4583        Q   Third   
6         0       1    male  54.0      0      0  51.8625        S   First   
7         0       3    male   2.0      3      1  21.0750        S   Third   
8         1       3  female  27.0      0      2  11.1333        S   Third   
9         1       2  female  14.0      1      0  30.0708        C  Second   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes

| Kolumna | Znaczenie |
|---|---|
| `survived` | Czy pasażer przeżył: `0` = nie, `1` = tak |
| `pclass` | Klasa biletu: `1` = pierwsza, `2` = druga, `3` = trzecia |
| `sex` | Płeć pasażera: `male` = mężczyzna, `female` = kobieta |
| `age` | Wiek pasażera |
| `sibsp` | Liczba rodzeństwa lub małżonków na statku |
| `parch` | Liczba rodziców lub dzieci na statku |
| `fare` | Cena biletu |
| `embarked` | Port wejścia na statek zapisany skrótem (`S` - Southampton, `C` - Cherbourg, `Q` - Queenstown)|
| `class` | Klasa biletu zapisana słownie: `First`, `Second`, `Third` |
| `who` | Typ osoby: `man`, `woman` lub `child` |
| `adult_male` | Czy pasażer był dorosłym mężczyzną: `True` / `False` |
| `deck` | Pokład/statkowa sekcja, np. `A`, `B`, `C`; często brakuje tej wartości |
| `embark_town` | Miasto/port wejścia na statek |
| `alive` | Czy pasażer przeżył zapisane tekstowo: `yes` / `no` |
| `alone` | Czy pasażer podróżował sam: `True` / `False` |

In [32]:
from pandas.io.formats.style_render import Subset
print("\n Brakujące wartości w każdej kolumie: ")
print(titanic.isnull().sum())

liczba_wierszy_przed_czyszczeniem = len(titanic)
titanic_clean = titanic.dropna(subset=["age"])
liczba_wierszy_po_czyszczeniu = len(titanic_clean)
print("="*50)
print("\n Liczba wierszy przed czyszczeniem: ", liczba_wierszy_przed_czyszczeniem)
print("Liczba wierszy po czyszczeniu: ", liczba_wierszy_po_czyszczeniu)
print("Liczba wierszy usuniętych: ", liczba_wierszy_przed_czyszczeniem - liczba_wierszy_po_czyszczeniu)
print("="*50)
print(titanic_clean.isnull().sum())


 Brakujące wartości w każdej kolumie: 
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

 Liczba wierszy przed czyszczeniem:  891
Liczba wierszy po czyszczeniu:  714
Liczba wierszy usuniętych:  177
survived         0
pclass           0
sex              0
age              0
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           530
embark_town      2
alive            0
alone            0
dtype: int64


In [33]:
#sekcja 3: ceny biletów

def categorize_fare(fare, fare_ranges):
    if fare <= fare_ranges[0]:
        return 0
    elif fare <= fare_ranges[1]:
        return 1
    else:
        return 2


min_fare = titanic_clean["fare"].min()
max_fare = titanic_clean["fare"].max()


print("Min fare: ", min_fare)
print("Max fare: ", max_fare)
fare_step = (max_fare - min_fare) / 3
fare_ranges = [
    min_fare + fare_step,
    min_fare + 2 * fare_step,
]

print("Krok fare: ", fare_step)


# Wyświetlenie przedziałów cenowych
print("\nPrzedziały cenowe biletów:")
print(f"Niska: {min_fare:.2f} - {fare_ranges[0]:.2f}")
print(f"Średnia: {fare_ranges[0]:.2f} - {fare_ranges[1]:.2f}")
print(f"Wysoka: {fare_ranges[1]:.2f} - {max_fare:.2f}")


Min fare:  0.0
Max fare:  512.3292
Krok fare:  170.7764

Przedziały cenowe biletów:
Niska: 0.00 - 170.78
Średnia: 170.78 - 341.55
Wysoka: 341.55 - 512.33


In [35]:
# * Które kolumny naprawdę mogą pomóc przewidzieć, czy pasażer przeżył?

# Po co wybieramy tylko niektóre cechy?

# Bo nie każda kolumna pomaga modelowi, bo niektóre dane są:
# - przydatne, bo mają związek z przeżyciem,
# - zbędne, bo powtarzają informacje z innych kolumn,
# - problematyczne, bo mają dużo braków,
# - zakazane, bo zawierają odpowiedź.

# [najważnijesze cechy: sex, pcalss, age, fare_category (fare zmienione na nasze kategorie)]

titanic_model = titanic_clean.copy()
titanic_model["sex"] = titanic_model["sex"].map({
    "male" : 1,
    "female" : 0
})
titanic_model["fare_category"] = titanic_model["fare"].apply(
    lambda fare: categorize_fare(fare, fare_ranges)
)
print(titanic_model["fare_category"].value_counts())
selected_features = ["sex", "pclass", "age", "fare_category"]
print(titanic_model[selected_features].head(10))



fare_category
0    696
1     15
2      3
Name: count, dtype: int64
    sex  pclass   age  fare_category
0     1       3  22.0              0
1     0       1  38.0              0
2     0       3  26.0              0
3     0       1  35.0              0
4     1       3  35.0              0
6     1       1  54.0              0
7     1       3   2.0              0
8     0       3  27.0              0
9     0       2  14.0              0
10    0       3   4.0              0
11    0       1  58.0              0
12    1       3  20.0              0
13    1       3  39.0              0
14    0       3  14.0              0
15    0       2  55.0              0
16    1       3   2.0              0
18    0       3  31.0              0
20    1       2  35.0              0
21    1       2  34.0              0
22    0       3  15.0              0
23    1       1  28.0              0
24    0       3   8.0              0
25    0       3  38.0              0
27    1       1  19.0              1
30    1 

In [37]:
x = titanic_model[selected_features]
y = titanic_model["survived"]
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42

)


scaler = StandardScaler()


x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)


# Sprawdzenie rozmiarów zbiorów
print("Rozmiary zbiorów:")
print(f"X_train: {x_train.shape}")
print(f"X_test: {x_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

Rozmiary zbiorów:
X_train: (571, 4)
X_test: (143, 4)
y_train: (571,)
y_test: (143,)


# **Jak działa skalowanie?**

| pasażer | wiek `age` | cena biletu `fare` |
| ------- | ---------: | -----------------: |
| A       |         10 |                  0 |
| B       |         20 |                100 |
| C       |         30 |                500 |


Cena biletu ma dużo większe liczby niż wiek, więc model mógłby za bardzo zwracać uwagę na fare.


| pasażer | `age` po skalowaniu | `fare` po skalowaniu |
| ------- | ------------------: | -------------------: |
| A       |               -1.22 |                -0.93 |
| B       |                0.00 |                -0.46 |
| C       |                1.22 |                 1.39 |


```
jak dane wypadają względenm swojej kolumny
```